# 06. Translation Quality Analysis for COCOLOFA-RU v2

Цель ноутбука: показать, что перевод `text_en -> text_ru` не является
неконтролируемым этапом пайплайна. Мы оцениваем перевод без LLM-judge:

1. corpus-level статистика статусов перевода;
2. дешевые sanity checks: длины, пустые строки, кириллица/латиница, экстремальные ratio;
3. task-specific proxies сохранения аргументативной структуры;
4. optional cross-lingual semantic similarity через sentence embeddings;
5. optional reference-free MT Quality Estimation через COMETKiwi.

Важно: BLEU/chrF здесь не являются основными метриками, потому что для корпуса
нет эталонного русского human reference-перевода. Поэтому основной честный режим —
reference-free QE и структурные прокси.

## Как запускать

Базовые ячейки работают без тяжелых дополнительных библиотек. Optional-ячеки
требуют установки пакетов в текущий conda env.

Рекомендуемый порядок:

1. Выполнить базовый анализ до раздела `Optional semantic similarity`.
2. Если нужна более сильная оценка, установить `sentence-transformers`.
3. Если нужна академически более сильная MT QE, установить `unbabel-comet`.
4. В конце сохранить `summary.md`, таблицы и список подозрительных примеров.

In [ ]:
# Optional dependencies. Run only if needed in your conda env.
# %pip install -q sentence-transformers
# %pip install -q unbabel-comet
# %pip install -q sacrebleu evaluate

In [ ]:
import json
import math
import re
from collections import Counter
from pathlib import Path
from statistics import mean

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 350)

In [ ]:
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "cocolofa_ru_v2.jsonl").exists()), Path.cwd())
DATASET_PATH = PROJECT_ROOT / "data" / "cocolofa_ru_v2.jsonl"
OUTPUT_DIR = PROJECT_ROOT / "translation_quality_analysis"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

for path in [OUTPUT_DIR, FIGURES_DIR, TABLES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}"

rows = [json.loads(line) for line in DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
df = pd.DataFrame(rows)

REQUIRED_COLUMNS = {"sample_id", "text_en", "text_ru", "label_str", "split", "translation_status"}
missing = sorted(REQUIRED_COLUMNS - set(df.columns))
assert not missing, f"Missing columns: {missing}"

df["sample_id"] = pd.to_numeric(df["sample_id"], errors="coerce").astype(int)
df["label_str"] = df["label_str"].astype(str).str.strip().str.lower()
df["split"] = df["split"].astype(str).str.strip().str.lower()
df["translation_status"] = df["translation_status"].astype(str).str.strip().str.lower()
df["text_en"] = df["text_en"].fillna("").astype(str)
df["text_ru"] = df["text_ru"].fillna("").astype(str)

ACCEPTED_STATUSES = {"ok", "repaired_ok"}
df["is_accepted"] = df["translation_status"].isin(ACCEPTED_STATUSES)

{
    "rows": len(df),
    "accepted_rows": int(df["is_accepted"].sum()),
    "accepted_rate": float(df["is_accepted"].mean()),
    "dataset_path": str(DATASET_PATH),
    "output_dir": str(OUTPUT_DIR),
}

## 1. Статусы перевода

Это первый уровень контроля качества. Он показывает, какая доля корпуса была
принята автоматически, какая потребовала repair, и какая отправлена в manual review.
Для ВКР это полезно как формальное описание accepted subset.

In [ ]:
status_summary = (
    df.groupby("translation_status", as_index=False)
    .agg(row_count=("sample_id", "count"))
    .assign(share=lambda x: x["row_count"] / len(df))
    .sort_values("row_count", ascending=False)
)

status_by_label = pd.crosstab(
    df["label_str"],
    df["translation_status"],
    margins=True,
)

accepted_by_label = (
    df.groupby("label_str", as_index=False)
    .agg(
        row_count=("sample_id", "count"),
        accepted_count=("is_accepted", "sum"),
        accepted_rate=("is_accepted", "mean"),
    )
    .sort_values("accepted_rate")
)

display(status_summary)
display(accepted_by_label)

status_summary.to_csv(TABLES_DIR / "translation_status_summary.csv", index=False)
accepted_by_label.to_csv(TABLES_DIR / "accepted_rate_by_label.csv", index=False)
status_by_label.to_csv(TABLES_DIR / "translation_status_by_label.csv")

In [ ]:
plt.figure(figsize=(8, 4))
ax = sns.barplot(data=status_summary, x="translation_status", y="row_count", color="#3B6EA8")
ax.set_title("Translation status distribution")
ax.set_xlabel("translation_status")
ax.set_ylabel("row count")
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "translation_status_distribution.png", dpi=200)
plt.show()

## 2. Базовые sanity metrics

Эти метрики не доказывают идеальный перевод, но быстро находят технические
проблемы: пустые строки, непереведенный английский текст, экстремальные изменения
длины, подозрительно малое количество кириллицы или слишком высокий остаток латиницы.

In [ ]:
CYRILLIC_RE = re.compile(r"[А-Яа-яЁё]")
LATIN_RE = re.compile(r"[A-Za-z]")
WORD_RE = re.compile(r"\w+", flags=re.UNICODE)
SENT_SPLIT_RE = re.compile(r"[.!?。！？]+")


def safe_ratio(num: float, den: float) -> float:
    return float(num / den) if den else 0.0


def count_sentences(text: str) -> int:
    parts = [part.strip() for part in SENT_SPLIT_RE.split(str(text)) if part.strip()]
    return max(1, len(parts)) if str(text).strip() else 0


def char_share(pattern: re.Pattern, text: str) -> float:
    text = str(text)
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return 0.0
    return len(pattern.findall(text)) / len(letters)


analysis_df = df.copy()
analysis_df["en_chars"] = analysis_df["text_en"].str.len()
analysis_df["ru_chars"] = analysis_df["text_ru"].str.len()
analysis_df["char_ratio_ru_en"] = analysis_df.apply(lambda r: safe_ratio(r["ru_chars"], r["en_chars"]), axis=1)
analysis_df["en_words"] = analysis_df["text_en"].map(lambda x: len(WORD_RE.findall(x)))
analysis_df["ru_words"] = analysis_df["text_ru"].map(lambda x: len(WORD_RE.findall(x)))
analysis_df["word_ratio_ru_en"] = analysis_df.apply(lambda r: safe_ratio(r["ru_words"], r["en_words"]), axis=1)
analysis_df["en_sentences"] = analysis_df["text_en"].map(count_sentences)
analysis_df["ru_sentences"] = analysis_df["text_ru"].map(count_sentences)
analysis_df["sentence_ratio_ru_en"] = analysis_df.apply(lambda r: safe_ratio(r["ru_sentences"], r["en_sentences"]), axis=1)
analysis_df["ru_cyrillic_share"] = analysis_df["text_ru"].map(lambda x: char_share(CYRILLIC_RE, x))
analysis_df["ru_latin_share"] = analysis_df["text_ru"].map(lambda x: char_share(LATIN_RE, x))

analysis_df["flag_empty_translation"] = analysis_df["text_ru"].str.strip().eq("")
analysis_df["flag_low_cyrillic"] = analysis_df["ru_cyrillic_share"] < 0.70
analysis_df["flag_high_latin"] = analysis_df["ru_latin_share"] > 0.20
analysis_df["flag_extreme_char_ratio"] = ~analysis_df["char_ratio_ru_en"].between(0.45, 2.20)
analysis_df["flag_extreme_sentence_ratio"] = ~analysis_df["sentence_ratio_ru_en"].between(0.40, 2.50)

sanity_columns = [
    "flag_empty_translation",
    "flag_low_cyrillic",
    "flag_high_latin",
    "flag_extreme_char_ratio",
    "flag_extreme_sentence_ratio",
]
analysis_df["sanity_flag_count"] = analysis_df[sanity_columns].sum(axis=1)
analysis_df["has_sanity_warning"] = analysis_df["sanity_flag_count"] > 0

sanity_summary = pd.DataFrame(
    [
        {
            "metric": col,
            "count": int(analysis_df[col].sum()),
            "share": float(analysis_df[col].mean()),
        }
        for col in sanity_columns
    ]
)

numeric_summary = analysis_df[
    [
        "char_ratio_ru_en",
        "word_ratio_ru_en",
        "sentence_ratio_ru_en",
        "ru_cyrillic_share",
        "ru_latin_share",
    ]
].describe(percentiles=[0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99]).T

display(sanity_summary)
display(numeric_summary)

sanity_summary.to_csv(TABLES_DIR / "sanity_flag_summary.csv", index=False)
numeric_summary.to_csv(TABLES_DIR / "translation_numeric_sanity_summary.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(analysis_df["char_ratio_ru_en"], bins=60, ax=axes[0], color="#3B6EA8")
axes[0].set_title("RU/EN character ratio")
axes[0].set_xlim(0, min(4, analysis_df["char_ratio_ru_en"].quantile(0.995)))

sns.histplot(analysis_df["sentence_ratio_ru_en"], bins=40, ax=axes[1], color="#629460")
axes[1].set_title("RU/EN sentence ratio")
axes[1].set_xlim(0, min(4, analysis_df["sentence_ratio_ru_en"].quantile(0.995)))

sns.histplot(analysis_df["ru_cyrillic_share"], bins=40, ax=axes[2], color="#C75C5C")
axes[2].set_title("Cyrillic share in Russian translation")
axes[2].set_xlim(0, 1)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "translation_sanity_distributions.png", dpi=200)
plt.show()

## 3. Task-specific proxies сохранения аргументативной структуры

Это не LLM-judge и не полноценная семантическая валидация. Это проверка
того, что в переводе не исчезли важные маркеры аргументации:

- отрицание;
- причинно-следственные связки;
- выводные маркеры;
- модальность долженствования;
- обобщения и большинство;
- альтернативы;
- маркеры эскалации для slippery slope.

Метрика считается только для тех строк, где соответствующий marker group
присутствовал в английском исходнике.

In [ ]:
MARKER_GROUPS = {
    "negation": {
        "en": [r"\bno\b", r"\bnot\b", r"\bnever\b", r"\bnobody\b", r"\bnothing\b", r"\bnone\b"],
        "ru": [r"\bне\b", r"\bнет\b", r"\bникогда\b", r"\bникто\b", r"\bничего\b"],
    },
    "causal": {
        "en": [r"\bbecause\b", r"\bsince\b", r"\bdue to\b", r"\bleads? to\b", r"\btherefore\b", r"\bso\b"],
        "ru": [r"\bпотому что\b", r"\bтак как\b", r"\bиз-за\b", r"\bприводит? к\b", r"\bследовательно\b", r"\bпоэтому\b"],
    },
    "conclusion": {
        "en": [r"\btherefore\b", r"\bthus\b", r"\bso\b", r"\bmeans? that\b", r"\bthis shows\b"],
        "ru": [r"\bследовательно\b", r"\bтаким образом\b", r"\bпоэтому\b", r"\bзначит\b", r"\bэто показывает\b"],
    },
    "modality": {
        "en": [r"\bshould\b", r"\bmust\b", r"\bneed to\b", r"\bhave to\b", r"\bought to\b"],
        "ru": [r"\bследует\b", r"\bдолжн", r"\bнужно\b", r"\bнеобходимо\b", r"\bобязан"],
    },
    "majority_quantifier": {
        "en": [r"\bmost\b", r"\bmajority\b", r"\beveryone\b", r"\ball\b", r"\bmany people\b"],
        "ru": [r"\bбольшинство\b", r"\bвсе\b", r"\bкаждый\b", r"\bмногие\b", r"\bмного людей\b"],
    },
    "alternative": {
        "en": [r"\beither\b", r"\bor\b", r"\bonly two\b", r"\bchoice\b", r"\boption\b"],
        "ru": [r"\bлибо\b", r"\bили\b", r"\bтолько два\b", r"\bвыбор\b", r"\bвариант"],
    },
    "escalation": {
        "en": [r"\bwill lead\b", r"\bthen\b", r"\bnext\b", r"\beventually\b", r"\btoo late\b", r"\bslippery\b"],
        "ru": [r"\bпривед[её]т\b", r"\bзатем\b", r"\bпотом\b", r"\bв итоге\b", r"\bслишком поздно\b"],
    },
    "authority": {
        "en": [r"\bexpert\b", r"\bscientist\b", r"\bstudy\b", r"\breport\b", r"\baccording to\b", r"\bsays?\b"],
        "ru": [r"\bэксперт", r"\bуч[её]н", r"\bисследован", r"\bотч[её]т", r"\bсогласно\b", r"\bговор"],
    },
    "tradition": {
        "en": [r"\btradition\b", r"\btraditional\b", r"\balways\b", r"\bused to\b", r"\bfor generations\b"],
        "ru": [r"\bтрадиц", r"\bтрадиционный\b", r"\bвсегда\b", r"\bпринято\b", r"\bпоколен"],
    },
    "nature": {
        "en": [r"\bnatural\b", r"\bnature\b", r"\bunnatural\b", r"\borganic\b"],
        "ru": [r"\bестествен", r"\bприрод", r"\bнеестествен", r"\bорганическ"],
    },
}


def has_any(patterns: list[str], text: str) -> bool:
    text = str(text).lower()
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in patterns)


marker_rows = []
for group, patterns in MARKER_GROUPS.items():
    en_col = f"marker_en_{group}"
    ru_col = f"marker_ru_{group}"
    preserved_col = f"marker_preserved_{group}"
    analysis_df[en_col] = analysis_df["text_en"].map(lambda x: has_any(patterns["en"], x))
    analysis_df[ru_col] = analysis_df["text_ru"].map(lambda x: has_any(patterns["ru"], x))
    analysis_df[preserved_col] = np.where(analysis_df[en_col], analysis_df[ru_col], np.nan)
    relevant = analysis_df[analysis_df[en_col]]
    marker_rows.append(
        {
            "marker_group": group,
            "source_count": int(len(relevant)),
            "preserved_count": int(relevant[ru_col].sum()),
            "preservation_rate": float(relevant[ru_col].mean()) if len(relevant) else np.nan,
        }
    )

marker_summary = pd.DataFrame(marker_rows).sort_values("preservation_rate")
preserved_cols = [f"marker_preserved_{group}" for group in MARKER_GROUPS]
analysis_df["argument_marker_preservation_score"] = analysis_df[preserved_cols].mean(axis=1, skipna=True)
analysis_df["argument_marker_source_count"] = analysis_df[[f"marker_en_{group}" for group in MARKER_GROUPS]].sum(axis=1)

display(marker_summary)
display(
    analysis_df.loc[analysis_df["argument_marker_source_count"] > 0, "argument_marker_preservation_score"]
    .describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
)

marker_summary.to_csv(TABLES_DIR / "argument_marker_preservation_summary.csv", index=False)

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = marker_summary.dropna(subset=["preservation_rate"]).sort_values("preservation_rate")
ax = sns.barplot(data=plot_df, y="marker_group", x="preservation_rate", color="#3B6EA8")
ax.set_title("Argument marker preservation proxies")
ax.set_xlabel("preservation rate")
ax.set_ylabel("marker group")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "argument_marker_preservation.png", dpi=200)
plt.show()

## 4. Suspicious examples

Объединяем технические sanity flags и структурные marker flags. Эти строки
не обязательно ошибочные, но они должны попасть в ручную выборочную проверку.

In [ ]:
analysis_df["flag_low_marker_preservation"] = (
    (analysis_df["argument_marker_source_count"] >= 2)
    & (analysis_df["argument_marker_preservation_score"].fillna(1.0) < 0.50)
)
analysis_df["translation_quality_flag_count"] = (
    analysis_df[sanity_columns].sum(axis=1)
    + analysis_df["flag_low_marker_preservation"].astype(int)
)
analysis_df["has_translation_quality_warning"] = analysis_df["translation_quality_flag_count"] > 0

suspicious_columns = [
    "sample_id",
    "split",
    "label_str",
    "translation_status",
    "translation_quality_flag_count",
    "sanity_flag_count",
    "flag_low_marker_preservation",
    "char_ratio_ru_en",
    "sentence_ratio_ru_en",
    "ru_cyrillic_share",
    "ru_latin_share",
    "argument_marker_source_count",
    "argument_marker_preservation_score",
    "text_en",
    "text_ru",
]

suspicious_df = (
    analysis_df[analysis_df["has_translation_quality_warning"]]
    .sort_values(["translation_quality_flag_count", "sanity_flag_count", "argument_marker_preservation_score"], ascending=[False, False, True])
    [suspicious_columns]
    .reset_index(drop=True)
)

warning_summary = pd.DataFrame(
    {
        "metric": [
            "rows_with_any_warning",
            "rows_with_sanity_warning",
            "rows_with_low_marker_preservation",
        ],
        "count": [
            int(analysis_df["has_translation_quality_warning"].sum()),
            int(analysis_df["has_sanity_warning"].sum()),
            int(analysis_df["flag_low_marker_preservation"].sum()),
        ],
        "share": [
            float(analysis_df["has_translation_quality_warning"].mean()),
            float(analysis_df["has_sanity_warning"].mean()),
            float(analysis_df["flag_low_marker_preservation"].mean()),
        ],
    }
)

display(warning_summary)
display(suspicious_df.head(30))

warning_summary.to_csv(TABLES_DIR / "translation_warning_summary.csv", index=False)
suspicious_df.to_csv(TABLES_DIR / "suspicious_translation_examples.csv", index=False)

## 5. Optional semantic similarity через cross-lingual embeddings

Это автоматическая оценка без reference-перевода и без LLM-judge. Идея:
английский исходник и русский перевод кодируются одной многоязычной sentence embedding
моделью, затем считается cosine similarity.

Рекомендуемые модели:

- `sentence-transformers/LaBSE` — сильный вариант для translation-pair similarity;
- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` — быстрее, но слабее.

Запускайте на sample или на GPU, если полный корпус считается долго.

In [ ]:
RUN_SEMANTIC_SIMILARITY = False
SEMANTIC_MODEL_NAME = "sentence-transformers/LaBSE"
SEMANTIC_SAMPLE_SIZE = 1200  # Set None for full corpus.
SEMANTIC_BATCH_SIZE = 32

if RUN_SEMANTIC_SIMILARITY:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import paired_cosine_distances

    semantic_df = analysis_df.copy()
    if SEMANTIC_SAMPLE_SIZE is not None and len(semantic_df) > SEMANTIC_SAMPLE_SIZE:
        semantic_df = semantic_df.sample(n=SEMANTIC_SAMPLE_SIZE, random_state=42).reset_index(drop=True)

    model = SentenceTransformer(SEMANTIC_MODEL_NAME)
    en_emb = model.encode(
        semantic_df["text_en"].tolist(),
        batch_size=SEMANTIC_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    ru_emb = model.encode(
        semantic_df["text_ru"].tolist(),
        batch_size=SEMANTIC_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    semantic_df["crosslingual_similarity"] = 1.0 - paired_cosine_distances(en_emb, ru_emb)
    semantic_df["flag_low_semantic_similarity"] = semantic_df["crosslingual_similarity"] < semantic_df["crosslingual_similarity"].quantile(0.05)

    display(semantic_df["crosslingual_similarity"].describe(percentiles=[0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99]))
    display(
        semantic_df.sort_values("crosslingual_similarity")
        [["sample_id", "label_str", "translation_status", "crosslingual_similarity", "text_en", "text_ru"]]
        .head(25)
    )

    semantic_df.to_csv(TABLES_DIR / "semantic_similarity_scores.csv", index=False)

    plt.figure(figsize=(8, 4))
    sns.histplot(semantic_df["crosslingual_similarity"], bins=50, color="#3B6EA8")
    plt.title(f"Cross-lingual semantic similarity: {SEMANTIC_MODEL_NAME}")
    plt.xlabel("cosine similarity")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "crosslingual_semantic_similarity.png", dpi=200)
    plt.show()
else:
    display(Markdown("Semantic similarity is disabled. Set `RUN_SEMANTIC_SIMILARITY = True` to run it."))

## 6. Optional COMETKiwi reference-free QE

COMETKiwi — learned quality estimation metric. Она не требует эталонного
русского перевода: на вход подается `src` и `mt`, модель возвращает score
качества перевода. Это самый сильный автоматический вариант без LLM-judge.

Рекомендуется сначала прогнать sample, например 500-1000 строк, потому что
модель тяжелее базовых sanity checks.

Важно: `Unbabel/wmt22-cometkiwi-da` — gated Hugging Face model. Перед запуском
нужно открыть страницу модели, принять условия лицензии и выполнить
`huggingface-cli login` в том же conda env, где открыт notebook.

In [ ]:
# Optional Hugging Face auth check for gated COMETKiwi models.
RUN_HF_AUTH_CHECK = False

if RUN_HF_AUTH_CHECK:
    from huggingface_hub import whoami

    try:
        user_info = whoami()
        display({"hf_user": user_info.get("name"), "auth": "ok"})
    except Exception as exc:
        display(
            Markdown(
                "Hugging Face auth is not configured for this environment. "
                "Run `huggingface-cli login` or set `HF_TOKEN`, then accept the model license: "
                "https://huggingface.co/Unbabel/wmt22-cometkiwi-da"
            )
        )
        raise exc
else:
    display(Markdown("HF auth check skipped. Set `RUN_HF_AUTH_CHECK = True` before COMETKiwi if needed."))

In [ ]:
RUN_COMETKIWI = False
COMET_MODEL_NAME = "Unbabel/wmt22-cometkiwi-da"
COMET_SAMPLE_SIZE = 1000  # Set None for full corpus.
COMET_BATCH_SIZE = 8
COMET_NUM_WORKERS = 1  # Avoid COMET/PyTorch MPS DataLoader bug on macOS.

if RUN_COMETKIWI:
    import torch
    from comet import download_model, load_from_checkpoint

    comet_df = analysis_df.copy()
    if COMET_SAMPLE_SIZE is not None and len(comet_df) > COMET_SAMPLE_SIZE:
        comet_df = comet_df.sample(n=COMET_SAMPLE_SIZE, random_state=42).reset_index(drop=True)

    try:
        model_path = download_model(COMET_MODEL_NAME)
    except Exception as exc:
        display(
            Markdown(
                f"Could not download `{COMET_MODEL_NAME}`. "
                "Most likely this is a Hugging Face gated-repo/auth issue. "
                "Open https://huggingface.co/Unbabel/wmt22-cometkiwi-da, accept the license, "
                "then run `huggingface-cli login` in the active conda env. "
                f"Original error: `{type(exc).__name__}: {exc}`"
            )
        )
        raise

    comet_model = load_from_checkpoint(model_path)
    data = [
        {"src": row["text_en"], "mt": row["text_ru"]}
        for row in comet_df[["text_en", "text_ru"]].to_dict("records")
    ]
    gpus = 1 if torch.cuda.is_available() else 0
    try:
        output = comet_model.predict(
            data,
            batch_size=COMET_BATCH_SIZE,
            gpus=gpus,
            num_workers=COMET_NUM_WORKERS,
        )
    except ValueError as exc:
        if "multiprocessing_context can only be used" not in str(exc):
            raise

        # Workaround for a COMET/PyTorch interaction on macOS where MPS is
        # detected and DataLoader receives multiprocessing_context with
        # num_workers=0 internally.
        original_mps_is_available = torch.backends.mps.is_available
        torch.backends.mps.is_available = lambda: False
        try:
            output = comet_model.predict(
                data,
                batch_size=COMET_BATCH_SIZE,
                gpus=0,
                accelerator="cpu",
                num_workers=0,
            )
        finally:
            torch.backends.mps.is_available = original_mps_is_available
    scores = getattr(output, "scores", output[0] if isinstance(output, tuple) else output)

    comet_df["cometkiwi_score"] = list(map(float, scores))
    threshold = comet_df["cometkiwi_score"].quantile(0.05)
    comet_df["flag_low_cometkiwi"] = comet_df["cometkiwi_score"] < threshold

    display(comet_df["cometkiwi_score"].describe(percentiles=[0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99]))
    display(
        comet_df.sort_values("cometkiwi_score")
        [["sample_id", "label_str", "translation_status", "cometkiwi_score", "text_en", "text_ru"]]
        .head(25)
    )

    comet_df.to_csv(TABLES_DIR / "cometkiwi_scores.csv", index=False)

    plt.figure(figsize=(8, 4))
    sns.histplot(comet_df["cometkiwi_score"], bins=50, color="#3B6EA8")
    plt.title(f"COMETKiwi reference-free QE: {COMET_MODEL_NAME}")
    plt.xlabel("COMETKiwi score")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "cometkiwi_scores.png", dpi=200)
    plt.show()
else:
    display(Markdown("COMETKiwi is disabled. Set `RUN_COMETKIWI = True` to run it."))

## 7. Reference-based metrics if human references appear later

BLEU/chrF можно использовать только если появится `text_ru_ref`, то есть
эталонный русский перевод. Для текущего корпуса это не основной режим.

Ниже оставлен готовый блок для `evaluate`/`sacrebleu`, чтобы позднее быстро
посчитать классические MT-метрики на небольшой human-reference подвыборке.

In [ ]:
RUN_REFERENCE_METRICS = False

if RUN_REFERENCE_METRICS:
    assert "text_ru_ref" in analysis_df.columns, "Need a human reference column: text_ru_ref"
    hypotheses = analysis_df["text_ru"].tolist()
    references = analysis_df["text_ru_ref"].tolist()

    try:
        import evaluate

        sacrebleu_metric = evaluate.load("sacrebleu")
        chrf_metric = evaluate.load("chrf")
        bleu = sacrebleu_metric.compute(predictions=hypotheses, references=[[ref] for ref in references])
        chrf = chrf_metric.compute(predictions=hypotheses, references=references)
        display({"BLEU": bleu["score"], "chrF": chrf["score"], "library": "evaluate"})
    except Exception:
        import sacrebleu

        bleu = sacrebleu.corpus_bleu(hypotheses, [references])
        chrf = sacrebleu.corpus_chrf(hypotheses, [references])
        display({"BLEU": bleu.score, "chrF": chrf.score, "library": "sacrebleu"})
else:
    display(Markdown("Reference-based BLEU/chrF skipped: no human Russian references in this corpus."))

## 8. Thesis-ready summary

Эта ячейка сохраняет краткую сводку для раздела о подготовке данных.

In [ ]:
accepted_count = int(analysis_df["is_accepted"].sum())
accepted_rate = float(analysis_df["is_accepted"].mean())
manual_review_count = int((analysis_df["translation_status"] == "manual_review").sum())
repaired_count = int((analysis_df["translation_status"] == "repaired_ok").sum())
warning_count = int(analysis_df["has_translation_quality_warning"].sum())
warning_rate = float(analysis_df["has_translation_quality_warning"].mean())
marker_mean = float(
    analysis_df.loc[
        analysis_df["argument_marker_source_count"] > 0,
        "argument_marker_preservation_score",
    ].mean()
)

summary_md = f'''# Translation Quality Analysis Summary

Dataset: `{DATASET_PATH}`

## Corpus-level status

- Total rows: {len(analysis_df)}
- Accepted rows (`ok` + `repaired_ok`): {accepted_count} ({accepted_rate:.2%})
- `repaired_ok`: {repaired_count}
- `manual_review`: {manual_review_count}

## Automatic sanity checks

- Rows with at least one automatic warning: {warning_count} ({warning_rate:.2%})
- Mean argument-marker preservation score among rows with source markers: {marker_mean:.3f}

## Methodological note

The corpus has no human Russian reference translations, therefore BLEU/chrF are not used
as primary translation-quality metrics. The analysis relies on automatic status filtering,
technical sanity checks, argument-marker preservation proxies, and optional reference-free
quality estimation through COMETKiwi.
'''

(OUTPUT_DIR / "translation_quality_summary.md").write_text(summary_md, encoding="utf-8")
display(Markdown(summary_md))

export_columns = [
    "sample_id",
    "split",
    "label_str",
    "translation_status",
    "is_accepted",
    "char_ratio_ru_en",
    "word_ratio_ru_en",
    "sentence_ratio_ru_en",
    "ru_cyrillic_share",
    "ru_latin_share",
    "argument_marker_source_count",
    "argument_marker_preservation_score",
    "translation_quality_flag_count",
    "has_translation_quality_warning",
]
analysis_df[export_columns].to_csv(TABLES_DIR / "translation_quality_scores_base.csv", index=False)